In [0]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  NOTEBOOK 02 — SILVER TRANSFORMATION                                    ║
# ║                                                                         ║
# ║  PIPELINE FLOW:                                                         ║
# ║                                                                         ║
# ║  Bronze tables (all StringType, may have schema_drift_flag=True)        ║
# ║       │                                                                 ║
# ║       ▼                                                                 ║
# ║  1 — COLUMN SELECTION & VALIDATION                                      ║
# ║       Keep only required columns (drop extras Bronze kept)              ║
# ║       Drop rows missing required column VALUES                          ║
# ║       │                                                                 ║
# ║       ▼                                                                 ║
# ║  2 — TYPE CASTING & CLEANING                                            ║
# ║       Cast StringType → proper types (DATE, INT, DOUBLE)               ║
# ║       trim/upper/lower standardisation                                  ║
# ║       │                                                                 ║
# ║       ▼                                                                 ║
# ║  3 — PII MASKING                                                        ║
# ║       email → ****@domain.com                                           ║
# ║       raw email dropped permanently                                     ║
# ║       │                                                                 ║
# ║       ▼                                                                 ║
# ║  4 — PRODUCT VALIDITY GATE                                              ║
# ║       Build "valid_products" set from dim_product_scd2                  ║
# ║       Only products with price IS NOT NULL AND > 0                      ║
# ║       Sales rows with invalid/missing price → DROPPED + logged          ║
# ║       │              │                                                  ║
# ║       ▼              ▼                                                  ║
# ║  5 — SCD2 (dims) + CLEAN WRITE (sales — valid rows only)               ║
# ║  Two-pass MERGE    fact_sales_clean ← valid price                       ║
# ║       fact_sales_clean ← valid rows only                                ║
# ║       │                                                                 ║
# ║       ▼                                                                 ║
# ║  6 — RUN TRACKING                                                       ║
# ║       pipeline_runs table updated with counts + duration                ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import logging
import sys
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, DateType,
    StringType, TimestampType, LongType
)
from pyspark.sql.window import Window

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

CATALOG  = "retail_data_project"
BRONZE   = f"{CATALOG}.01_bronze"
SILVER   = f"{CATALOG}.02_silver"
METADATA = f"{CATALOG}.04_metadata"

CUSTOMER_BRONZE  = f"{BRONZE}.customers"
PRODUCT_BRONZE   = f"{BRONZE}.products"
SALES_BRONZE     = f"{BRONZE}.sales"
CUSTOMER_SILVER  = f"{SILVER}.dim_customer_scd2"
PRODUCT_SILVER   = f"{SILVER}.dim_product_scd2"
SALES_CLEAN      = f"{SILVER}.fact_sales_clean"
PIPELINE_RUNS    = f"{METADATA}.pipeline_runs"

DATE_FORMAT = "yyyy-MM-dd"

CUSTOMER_REQUIRED_COLS = ["customer_id", "name", "email", "city", "signup_date"]
PRODUCT_REQUIRED_COLS  = ["product_id", "product_name", "category", "price"]
SALES_REQUIRED_COLS    = ["order_id", "customer_id", "product_id", "quantity", "order_date"]

BRONZE_META_COLS = ["received_date", "source_file", "ingestion_ts", "schema_drift_flag"]

# ══════════════════════════════════════════════════════════════════════════════
# STRUCTURED LOGGER
# ══════════════════════════════════════════════════════════════════════════════

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            "%Y-%m-%d %H:%M:%S"
        ))
        logger.addHandler(h)
        logger.setLevel(logging.INFO)
    return logger

logger    = get_logger("02_silver_transform")
RUN_START = datetime.now()
RUN_DATE  = str(datetime.now().date())
RUN_ID    = f"silver_{RUN_DATE}_{datetime.now().strftime('%H%M%S')}"

logger.info(f"Silver transform STARTED | run_id={RUN_ID}")

def record_run(status: str, rows: int, message: str) -> None:
    try:
        from pyspark.sql.types import StructType, StructField
        schema = StructType([
            StructField("run_id",         StringType(),    False),
            StructField("notebook_name",  StringType(),    False),
            StructField("run_date",       StringType(),    False),
            StructField("start_ts",       TimestampType(), False),
            StructField("end_ts",         TimestampType(), False),
            StructField("status",         StringType(),    False),
            StructField("rows_processed", LongType(),      False),
            StructField("message",        StringType(),    True),
        ])
        spark.createDataFrame(
            [(RUN_ID, "02_silver", RUN_DATE, RUN_START,
              datetime.now(), status, int(rows), str(message)[:500])],
            schema=schema
        ).write.format("delta").mode("append").saveAsTable(PIPELINE_RUNS)
    except Exception as e:
        logger.warning(f"[PIPELINE_RUNS] write failed: {e}")

record_run("RUNNING", 0, "Started")

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def select_required_columns(df: DataFrame, required_cols: list, table_name: str) -> DataFrame:
    available = set(df.columns)
    required  = set(required_cols)
    meta      = set(BRONZE_META_COLS)

    extra_in_bronze  = available - required - meta
    missing_required = required - available

    if extra_in_bronze:
        logger.warning(f"[COL_SELECT][{table_name}] Dropping extra columns: {extra_in_bronze}")

    cols_to_select = [c for c in (required_cols + BRONZE_META_COLS) if c in available]
    return df.select(*cols_to_select)

def drop_invalid_rows(df: DataFrame, not_null_cols: list, table_name: str) -> tuple:
    total_before = df.count()
    null_condition = F.lit(False)
    for c in not_null_cols:
        if c in df.columns:
            # Type-aware check: only check for empty strings on StringType columns to avoid CAST_INVALID_INPUT
            is_string = isinstance(df.schema[c].dataType, StringType)
            if is_string:
                null_condition = null_condition | (F.col(c).isNull() | (F.col(c) == "") | (F.col(c) == "NULL"))
            else:
                null_condition = null_condition | F.col(c).isNull()

    df_valid = df.filter(~null_condition)
    invalid_count = total_before - df_valid.count()

    if invalid_count > 0:
        logger.warning(f"[ROW_VALIDATION][{table_name}] Dropped {invalid_count:,} rows with NULL/empty required values.")
    return df_valid, invalid_count, total_before

def dedup_latest(df: DataFrame, partition_col: str) -> DataFrame:
    w = Window.partitionBy(partition_col).orderBy(F.col("ingestion_ts").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — DIM_CUSTOMER
# ══════════════════════════════════════════════════════════════════════════════

logger.info(f"\n{'═'*62}\n  PROCESSING: dim_customer_scd2\n{'═'*62}")
df_cust_raw = spark.table(CUSTOMER_BRONZE)
df_cust = select_required_columns(df_cust_raw, CUSTOMER_REQUIRED_COLS, "customers")
df_cust, cust_dropped, cust_total = drop_invalid_rows(df_cust, ["customer_id"], "customers")

df_cust = (
    df_cust
    .withColumn("name",        F.trim(F.col("name")))
    .withColumn("email",       F.lower(F.trim(F.col("email"))))
    .withColumn("city",        F.upper(F.trim(F.col("city"))))
    .withColumn("signup_date", F.to_date(F.col("signup_date"), DATE_FORMAT))
)

df_cust = dedup_latest(df_cust, "customer_id")
df_cust = (
    df_cust
    .withColumn("email_masked", 
        F.when(F.col("email").isNotNull() & (F.col("email") != ""),
               F.concat(F.lit("****@"), F.regexp_replace(F.col("email"), r"^[^@]+@", "")))
        .otherwise(F.lit(None).cast(StringType())))
    .drop("email")
)

df_cust = df_cust.withColumn("change_hash", 
    F.md5(F.concat_ws("||", F.coalesce(F.col("name"), F.lit("")), 
    F.coalesce(F.col("email_masked"), F.lit("")), F.coalesce(F.col("city"), F.lit("")))))

df_cust.createOrReplaceTempView("incoming_customers")

spark.sql(f"MERGE INTO {CUSTOMER_SILVER} AS target USING incoming_customers AS source ON target.customer_id = source.customer_id AND target.is_current = true WHEN MATCHED AND target.change_hash != source.change_hash THEN UPDATE SET target.end_date = current_date(), target.is_current = false")
spark.sql(f"MERGE INTO {CUSTOMER_SILVER} AS target USING (SELECT s.*, current_date() AS start_date, CAST(NULL AS DATE) AS end_date, true AS is_current FROM incoming_customers s LEFT ANTI JOIN {CUSTOMER_SILVER} t ON t.customer_id = s.customer_id AND t.is_current = true) AS source ON target.customer_id = source.customer_id AND target.is_current = true WHEN NOT MATCHED THEN INSERT *")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — DIM_PRODUCT
# ══════════════════════════════════════════════════════════════════════════════

logger.info(f"\n{'═'*62}\n  PROCESSING: dim_product_scd2\n{'═'*62}")
df_prod_raw = spark.table(PRODUCT_BRONZE)
df_prod = select_required_columns(df_prod_raw, PRODUCT_REQUIRED_COLS, "products")

# Initial check for product_id (still StringType here)
df_prod, prod_dropped, prod_total = drop_invalid_rows(df_prod, ["product_id"], "products")

# Casting
df_prod = (
    df_prod
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category",     F.upper(F.trim(F.col("category"))))
    .withColumn("price",        F.col("price").cast(DoubleType()))
)

# FIXED: numeric check for null_price_count (removed empty string comparison)
null_price_count = df_prod.filter(F.col("price").isNull()).count()
logger.info(f"[products] NULL price count: {null_price_count:,}")

df_prod = dedup_latest(df_prod, "product_id")
df_prod = df_prod.withColumn("change_hash", 
    F.md5(F.concat_ws("||", F.coalesce(F.col("product_name"), F.lit("")), 
    F.coalesce(F.col("category"), F.lit("")), 
    F.when(F.col("price").isNull(), F.lit("NULL")).otherwise(F.col("price").cast(StringType())))))

df_prod.createOrReplaceTempView("incoming_products")

spark.sql(f"MERGE INTO {PRODUCT_SILVER} AS target USING incoming_products AS source ON target.product_id = source.product_id AND target.is_current = true WHEN MATCHED AND target.change_hash != source.change_hash THEN UPDATE SET target.end_date = current_date(), target.is_current = false")
spark.sql(f"MERGE INTO {PRODUCT_SILVER} AS target USING (SELECT s.*, current_date() AS start_date, CAST(NULL AS DATE) AS end_date, true AS is_current FROM incoming_products s LEFT ANTI JOIN {PRODUCT_SILVER} t ON t.product_id = s.product_id AND t.is_current = true) AS source ON target.product_id = source.product_id AND target.is_current = true WHEN NOT MATCHED THEN INSERT *")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — FACT_SALES
# ══════════════════════════════════════════════════════════════════════════════

logger.info(f"\n{'═'*62}\n  PROCESSING: fact_sales\n{'═'*62}")
df_sales_raw = spark.table(SALES_BRONZE)
df_sales = select_required_columns(df_sales_raw, SALES_REQUIRED_COLS, "sales")
df_sales, sales_dropped, sales_total = drop_invalid_rows(df_sales, ["order_id", "customer_id", "product_id", "quantity", "order_date"], "sales")

df_sales = (
    df_sales
    .withColumn("quantity",   F.col("quantity").cast(IntegerType()))
    .withColumn("order_date", F.to_date(F.col("order_date"), DATE_FORMAT))
)

# Second drop pass for casted types
df_sales, cast_dropped, _ = drop_invalid_rows(df_sales, ["quantity", "order_date"], "sales_post_cast")
df_sales = dedup_latest(df_sales, "order_id")

# Product Validity Gate
df_all_products_current = spark.table(PRODUCT_SILVER).filter("is_current = true").select(
    "product_id", "price", F.lit(True).alias("is_current_in_dim"),
    F.when(F.col("price").isNotNull() & (F.col("price") > 0), F.lit("VALID"))
    .otherwise(F.lit("INVALID_PRICE")).alias("product_status")
)

df_sales_with_product = df_sales.join(df_all_products_current, on="product_id", how="left")
df_clean = df_sales_with_product.filter(F.col("product_status") == "VALID") \
           .withColumn("total_amount", F.col("quantity") * F.col("price"))

invalid_rows_dropped = df_sales_with_product.filter(F.col("product_status").isNull() | (F.col("product_status") != "VALID")).count()
logger.info(f"[sales] Clean: {df_clean.count():,} | Dropped (product gate): {invalid_rows_dropped:,}")

df_clean.createOrReplaceTempView("incoming_sales_clean")
spark.sql(f"MERGE INTO {SALES_CLEAN} AS target USING incoming_sales_clean AS source ON target.order_id = source.order_id WHEN NOT MATCHED THEN INSERT *")

# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

cust_curr = spark.table(CUSTOMER_SILVER).filter("is_current = true").count()
prod_curr = spark.table(PRODUCT_SILVER).filter("is_current = true").count()
final_clean = spark.table(SALES_CLEAN).count()

record_run("SUCCESS", final_clean, f"customers={cust_curr} products={prod_curr} sales={final_clean}")
logger.info(f"\n{'='*62}\n  SILVER TRANSFORMATION COMPLETE ✓\n{'='*62}")

In [0]:
%sql
select count(*) from retail_data_project.02_silver.dim_customer_scd2;

In [0]:
%sql
select count(*) from retail_data_project.02_silver.fact_sales_clean;

In [0]:
%sql
select count(*) from retail_data_project.02_silver.dim_product_scd2;

In [0]:
%sql
select * from retail_data_project.02_silver.dim_product_scd2 where product_id = "P00000";